# 🎯 LLM Activation Calibration System (Google Colab Multi-Node Pipeline)

This notebook provides a turnkey environment on **Google Colab (T4 / L4 / A100 GPU)** for **Decoupled Activation Calibration** of Edge LLMs.

### 💡 Key Workflow Advantages
1. **Decoupled 2-Stage Architecture**: Calibration is performed once; layer statistics (, , , ) are serialized as  artifacts to Google Drive. Downstream quantization configurations (W8A16, W4A16, W4A4, SpinQuant, etc.) load these artifacts in seconds without repeating forward passes.
2. **Dual-Node Parallel Execution (~35 min for all 5 models)**:
   * **Node 1**: Llama 3.2 1B () ➔ Gemma 2 2B () ➔ Llama 3.2 3B () ➔ **Total ~31 minutes**
   * **Node 2**: Qwen 3.5 2B () ➔ Qwen 3.5 4B () ➔ **Total ~34 minutes**
3. **Academic Golden Standard**: Strictly follows GPTQ / AWQ / QuaRot standards, sampling **128 sequences × 2048 tokens** from   split (262,144 tokens total, ).

---


## 1. Verify GPU Hardware (NVIDIA-SMI)


In [ ]:
!nvidia-smi

## 2. Mount Google Drive (Persistent Storage for .pt Artifacts)
Saves calibration artifacts directly to Google Drive to guard against session timeouts.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create calibration artifact directory
CALIB_OUTPUT_DIR = "/content/drive/MyDrive/model_quantization/calibration_results"
os.makedirs(CALIB_OUTPUT_DIR, exist_ok=True)
print(f"✅ Artifact directory ready: {CALIB_OUTPUT_DIR}")


## 3. Clone Repository & Install Dependencies


In [ ]:
import os

REPO_URL = "https://github.com/your-username/model_quantization.git"

if not os.path.exists("/content/model_quantization"):
    !git clone  /content/model_quantization
    %cd /content/model_quantization
else:
    %cd /content/model_quantization
    !git pull

# Install core dependencies
!pip install -q "transformers>=4.40.0" "accelerate>=0.28.0" "datasets>=2.18.0" sentencepiece protobuf scipy tqdm


## 4. Hugging Face Authentication (Required for LLaMA 3.2 & Gemma 2)
Authenticate using Colab Secrets to access gated model repositories safely.


In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    login(userdata.get('HF_TOKEN'))
    print("✅ Hugging Face authenticated via Colab Secrets!")
except Exception:
    # Fallback to interactive prompt if secret is not set:
    # login()
    pass


---
## 🚀 Node 1 Batch Execution
**Models**:  ➔  ➔   
**Estimated Runtime**: ~31 minutes


In [ ]:
# Node 1 Batch Calibration Script
CALIB_DIR = "/content/drive/MyDrive/model_quantization/calibration_results"

print("🚀 [1/3] Calibrating meta-llama/Llama-3.2-1B (~6 min) ...")
!python scripts/run_calibration.py --model_id meta-llama/Llama-3.2-1B --output_dir 

print("
🚀 [2/3] Calibrating google/gemma-2-2b-it (~8 min) ...")
!python scripts/run_calibration.py --model_id google/gemma-2-2b-it --output_dir 

print("
🚀 [3/3] Calibrating meta-llama/Llama-3.2-3B (~17 min) ...")
!python scripts/run_calibration.py --model_id meta-llama/Llama-3.2-3B --output_dir 

print("
🎉 Node 1 batch calibration completed successfully!")


---
## 🚀 Node 2 Batch Execution
**Models**:  ➔   
**Estimated Runtime**: ~34 minutes


In [ ]:
# Node 2 Batch Calibration Script
CALIB_DIR = "/content/drive/MyDrive/model_quantization/calibration_results"

print("🚀 [1/2] Calibrating Qwen/Qwen3.5-2B (~11 min) ...")
!python scripts/run_calibration.py --model_id Qwen/Qwen3.5-2B --output_dir 

print("
🚀 [2/2] Calibrating Qwen/Qwen3.5-4B (~23 min) ...")
!python scripts/run_calibration.py --model_id Qwen/Qwen3.5-4B --output_dir 

print("
🎉 Node 2 batch calibration completed successfully!")


---
## 📊 5. Calibration Inspection & Verification
Loads  artifacts from Google Drive to inspect layer statistics and clipping thresholds.


In [ ]:
import glob
import os
import torch
import pandas as pd

CALIB_DIR = "/content/drive/MyDrive/model_quantization/calibration_results"
pt_files = sorted(glob.glob(os.path.join(CALIB_DIR, "*.pt")))

print(f"📂 Found {len(pt_files)} calibration artifacts in storage:
")

summary_rows = []
for f in pt_files:
    data = torch.load(f, map_location="cpu")
    m_id = data["model_id"]
    num_layers = data["num_layers"]
    t_sec = data["calibration_time_sec"]
    
    layer_keys = list(data["layers"].keys())
    first_layer = data["layers"][layer_keys[0]]
    deep_layer = data["layers"][layer_keys[-1]]
    
    summary_rows.append({
        "Model": m_id,
        "Layers": num_layers,
        "Time": f"{t_sec:.1f}s ({t_sec/60:.1f}m)",
        "L0 Max": f"{first_layer['abs_max']:.2f}",
        "Deep Max": f"{deep_layer['abs_max']:.2f}",
        "Deep Clip (99.99%)": f"{deep_layer['clip_max_9999']:.2f}",
        "Deep INT8 Scale": f"{deep_layer['scales']['int8_sym_scale']:.6f}",
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)
